In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# Install downloader
!pip install -q gdown

# Download your file from Google Drive
import gdown

file_id = "1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq"
output = "/kaggle/working/zebramap_final.zip"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

# Extract the zip
import zipfile

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall("/kaggle/working/data")

print("✅ Download + Extraction Completed")

Downloading...
From (original): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq
From (redirected): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq&confirm=t&uuid=c2d8f3cf-f5e1-4133-b655-321f0dab42a7
To: /kaggle/working/zebramap_final.zip
100%|██████████| 10.5G/10.5G [01:46<00:00, 98.2MB/s]


OSError: [Errno 28] No space left on device

In [4]:
!ls -lh /kaggle/working/zebramap_final.zip

-rw-r--r-- 1 root root 9.8G Apr  7 14:43 /kaggle/working/zebramap_final.zip


In [5]:
import zipfile

zip_path = "/kaggle/working/zebramap_final.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    bad_file = z.testzip()

if bad_file is None:
    print("✅ ZIP file is valid (no corruption)")
else:
    print("❌ Corrupted file found:", bad_file)

✅ ZIP file is valid (no corruption)


In [6]:
import zipfile

with zipfile.ZipFile("/kaggle/working/zebramap_final.zip", 'r') as z:
    files = z.namelist()

print("Total files inside zip:", len(files))

Total files inside zip: 110262


In [8]:
print(files[:10])

['100/', '100/PMC10026180/', '100/PMC10026180/fped-11-1103726-g001.jpg', '100/PMC10026180/fped-11-1103726-g002.jpg', '100/PMC10026180/fped-11-1103726-g003.jpg', '100/PMC10026180/fped-11-1103726-g004.jpg', '100/PMC10026180/fped-11-1103726-g005.jpg', '100/PMC10026180/fped-11-1103726-g006.jpg', '100/PMC10026180/fped-11-1103726-g007.jpg', '100/PMC10026180/fped-11-1103726-g008.jpg']


In [5]:
import zipfile
import os

zip_path  = "/kaggle/working/zebramap_final.zip"
extract_to = "/kaggle/temp/images"

os.makedirs(extract_to, exist_ok=True)

print("Extracting ZIP to /kaggle/temp/...")
print("This will take 5-10 minutes...")

with zipfile.ZipFile(zip_path, 'r') as z:
    total = len(z.namelist())
    for i, member in enumerate(z.namelist()):
        z.extract(member, extract_to)
        if i % 5000 == 0:
            print(f"  {i}/{total} files extracted "
                  f"({i/total*100:.1f}%)")

print(f"\n✓ Extraction complete")

# Verify
sample_dirs = os.listdir(extract_to)[:5]
print(f"Sample folders: {sample_dirs}")
total_imgs = sum(
    len(files) for _, _, files in os.walk(extract_to)
)
print(f"Total images  : {total_imgs}")

Extracting ZIP to /kaggle/temp/...
This will take 5-10 minutes...
  0/110262 files extracted (0.0%)
  5000/110262 files extracted (4.5%)
  10000/110262 files extracted (9.1%)
  15000/110262 files extracted (13.6%)
  20000/110262 files extracted (18.1%)
  25000/110262 files extracted (22.7%)
  30000/110262 files extracted (27.2%)
  35000/110262 files extracted (31.7%)
  40000/110262 files extracted (36.3%)
  45000/110262 files extracted (40.8%)
  50000/110262 files extracted (45.3%)
  55000/110262 files extracted (49.9%)
  60000/110262 files extracted (54.4%)
  65000/110262 files extracted (59.0%)
  70000/110262 files extracted (63.5%)
  75000/110262 files extracted (68.0%)
  80000/110262 files extracted (72.6%)
  85000/110262 files extracted (77.1%)
  90000/110262 files extracted (81.6%)
  95000/110262 files extracted (86.2%)
  100000/110262 files extracted (90.7%)
  105000/110262 files extracted (95.2%)
  110000/110262 files extracted (99.8%)

✓ Extraction complete
Sample folders: ['1

In [11]:
import ast
import pandas as pd

def fix_path_kaggle(img_val):
    try:
        images = img_val if isinstance(img_val, list) \
                         else ast.literal_eval(img_val)
        fixed = []
        for img in images:
            old_path = img['path']
            if 'images/' in old_path:
                rel = old_path.split('images/')[-1]
                img['path'] = f"/kaggle/temp/images/{rel}"
            fixed.append(img)
        return fixed
    except:
        return []

# Reload CSV and fix paths
DATA = "/kaggle/working/data"
df   = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")
df['images_fixed'] = df['images'].apply(fix_path_kaggle)

# Verify paths
found = 0
for _, row in df.head(20).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found += 1

print(f"✓ Path fix complete")
print(f"  Found {found}/20 sample images")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/data/clean_multimodal_samples.csv'

In [1]:
import gdown
import os
import ast
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

os.makedirs("/kaggle/working/data", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# Download CSV
print("\nDownloading CSV...")
gdown.download(
    "https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye",
    "/kaggle/working/data/clean_multimodal_samples.csv",
    quiet=False
)

# Load and prepare data
df = pd.read_csv("/kaggle/working/data/clean_multimodal_samples.csv")
print(f"✓ CSV loaded: {df.shape}")

# Fix image paths to point to /kaggle/temp/images
def fix_path_kaggle(img_val):
    try:
        images = img_val if isinstance(img_val, list) \
                         else eval(img_val)
        fixed = []
        for img in images:
            old_path = img['path']
            if 'images/' in old_path:
                rel = old_path.split('images/')[-1]
                img['path'] = f"/kaggle/temp/images/{rel}"
            fixed.append(img)
        return fixed
    except:
        return []

print("Fixing image paths...")
df['images_fixed'] = df['images'].apply(fix_path_kaggle)

# Verify paths
found = 0
for _, row in df.head(20).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found += 1
print(f"✓ Image paths verified: {found}/20 found")

# Encode labels
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Add symptom_text
def make_symptom_text(sym_str):
    try:
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join([s.lower().strip() for s in syms])
    except:
        return str(sym_str)

df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

# Filter + split
label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"\n✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

# Final image check
found2 = 0
for _, row in train_df.head(20).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found2 += 1
print(f"  Images  : {found2}/20 train samples verified")

✓ Device: Tesla T4



Downloading...
From: https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye
To: /kaggle/working/data/clean_multimodal_samples.csv
100%|██████████| 28.2M/28.2M [00:00<00:00, 79.6MB/s]


✓ CSV loaded: (36487, 9)
Fixing image paths...
✓ Image paths verified: 0/20 found

✓ Data ready
  Train   : 29049
  Test    : 7263
  Classes : 1199
  Images  : 0/20 train samples verified


In [13]:
import os

# Delete the ZIP to free space
zip_path = "/kaggle/working/zebramap_final.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
    print("✓ ZIP deleted — space freed")
else:
    print("ZIP already gone")

# Check available space
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk space:")
print(f"  Total : {total//(1024**3)} GB")
print(f"  Used  : {used//(1024**3)} GB")
print(f"  Free  : {free//(1024**3)} GB")

✓ ZIP deleted — space freed

Disk space:
  Total : 19 GB
  Used  : 9 GB
  Free  : 9 GB


In [7]:
import os

# Check what's in temp
print("Contents of /kaggle/temp/:")
for item in os.listdir("/kaggle/temp"):
    full = os.path.join("/kaggle/temp", item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  [DIR]  {item}/  ({count} items)")
    else:
        size = os.path.getsize(full) / (1024*1024)
        print(f"  [FILE] {item}  ({size:.1f} MB)")

# Check one level deeper
print("\nContents of /kaggle/temp/images/ (first 5):")
img_path = "/kaggle/temp/images"
if os.path.exists(img_path):
    items = os.listdir(img_path)[:5]
    for item in items:
        full = os.path.join(img_path, item)
        if os.path.isdir(full):
            count = len(os.listdir(full))
            print(f"  [DIR]  {item}/  ({count} items)")
else:
    print("  /kaggle/temp/images does NOT exist")

# Check what a sample path looks like in the CSV
import pandas as pd
import ast

df = pd.read_csv("/kaggle/working/data/clean_multimodal_samples.csv")
sample_imgs = eval(df['images'].iloc[0])
print(f"\nSample path in CSV:")
print(f"  {sample_imgs[0]['path']}")

# Check what actual path looks like on disk
print(f"\nActual disk structure sample:")
for root, dirs, files in os.walk("/kaggle/temp"):
    for f in files[:3]:
        print(f"  {os.path.join(root, f)}")
    break

Contents of /kaggle/temp/:
  [DIR]  images/  (1331 items)

Contents of /kaggle/temp/images/ (first 5):
  [DIR]  1450/  (2 items)
  [DIR]  199/  (27 items)
  [DIR]  85102/  (15 items)
  [DIR]  381/  (7 items)
  [DIR]  2800/  (54 items)

Sample path in CSV:
  /content/drive/MyDrive/rare_disease_project/data/zebramap/images/758/PMC11809835/RomJOphthalmol-68-470-g001.jpg

Actual disk structure sample:


In [8]:
import os
import ast
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# Load CSV
df = pd.read_csv("/kaggle/working/data/clean_multimodal_samples.csv")

# Fix paths — old path has 'images/758/PMC.../file.jpg'
# New path should be '/kaggle/temp/images/758/PMC.../file.jpg'
def fix_path_kaggle(img_val):
    try:
        images = eval(img_val)
        fixed  = []
        for img in images:
            old = img['path']
            # Extract everything after 'zebramap/images/'
            if 'zebramap/images/' in old:
                rel = old.split('zebramap/images/')[-1]
            elif 'images/' in old:
                rel = old.split('images/')[-1]
            else:
                rel = old
            img['path'] = f"/kaggle/temp/images/{rel}"
            fixed.append(img)
        return fixed
    except:
        return []

print("Fixing paths...")
df['images_fixed'] = df['images'].apply(fix_path_kaggle)

# Verify
found = 0
for _, row in df.head(20).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found += 1
        if found <= 3:
            print(f"  ✓ {imgs[0]['path'][-60:]}")

print(f"\nFound: {found}/20 sample images")

# Encode labels
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Symptom text
def make_symptom_text(sym_str):
    try:
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join([s.lower().strip() for s in syms])
    except:
        return str(sym_str)

df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

# Filter + split
label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered, test_size=0.2,
    random_state=42, stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

# Final check
found2 = 0
for _, row in train_df.head(50).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found2 += 1

print(f"\n✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")
print(f"  Images  : {found2}/50 train samples verified")

✓ Device: Tesla T4
Fixing paths...
  ✓ e/temp/images/758/PMC11809835/RomJOphthalmol-68-470-g001.jpg
  ✓ mp/images/758/PMC11804138/10.1177_15385744241290007-fig1.jpg
  ✓ /kaggle/temp/images/758/PMC10334318/gr1.jpg

Found: 19/20 sample images

✓ Data ready
  Train   : 29049
  Test    : 7263
  Classes : 1199
  Images  : 41/50 train samples verified


In [9]:
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
import matplotlib.pyplot as plt
import json
from datetime import datetime

# ── Dataset ────────────────────────────────────────────────────
class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                images = row['images_fixed']
                if not isinstance(images, list):
                    images = eval(images)
                for img_info in images:
                    path = img_info['path']
                    if os.path.exists(path):
                        self.samples.append({
                            'path' : path,
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Building datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(cnn_train_ds, batch_size=64,
                               shuffle=True,  num_workers=4)
cnn_test_loader  = DataLoader(cnn_test_ds,  batch_size=64,
                               shuffle=False, num_workers=4)

print(f"✓ Datasets ready")
print(f"  Train images  : {len(cnn_train_ds)}")
print(f"  Test images   : {len(cnn_test_ds)}")
print(f"  Train batches : {len(cnn_train_loader)}")
print(f"  Classes       : {NUM_CLASSES}")

# ── Model ──────────────────────────────────────────────────────
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 15
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler  = CosineAnnealingLR(
    optimizer, T_max=CNN_EPOCHS, eta_min=1e-6)

cnn_losses = []
cnn_accs   = []

print(f"\nTraining CNN — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_losses.append(avg_loss)
    cnn_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ Training complete")

# ── Evaluation ─────────────────────────────────────────────────
def evaluate_topk(model, loader, device, k=5):
    model.eval()
    all_preds, all_labels = [], []
    top5_correct, total   = 0, 0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            logits = model(images)

            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            topk = logits.topk(k, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    top5_correct += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top5_accuracy": round(top5_correct/total*100, 2),
        "total_samples": total
    }

print("\nEvaluating...")
cnn_metrics = evaluate_topk(cnn_model, cnn_test_loader, device)

print("\n" + "=" * 55)
print("EXP 3 — CNN UPPER BOUND (Full Dataset)")
print("=" * 55)
print(f"  Accuracy     : {cnn_metrics['accuracy']}%")
print(f"  F1 Macro     : {cnn_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {cnn_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {cnn_metrics['top5_accuracy']}%")
print(f"  Test samples : {cnn_metrics['total_samples']}")

# ── Save ───────────────────────────────────────────────────────
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, "/kaggle/working/models/exp3_cnn_full.pt")
print(f"✓ Model saved")

# ── Plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_losses, color='#534AB7',
             linewidth=2, marker='o', markersize=3)
axes[0].set_title('CNN Loss — Full Dataset')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_accs, color='#BA7517',
             linewidth=2, marker='o', markersize=3)
axes[1].set_title('CNN Accuracy — Full Dataset')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/results/day5_cnn_curves.png",
            dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ────────────────────────────────────────────────────
nlp_metrics = {
    "accuracy": 16.74, "f1_macro": 2.71,
    "f1_weighted": 10.99, "top5_accuracy": 35.81,
    "total_samples": 7263
}

summary = {
    "day"        : 5,
    "experiment" : "exp3_full_upperbound",
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "status"     : "Day 5 complete"
}

with open("/kaggle/working/results/day5_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("DAY 5 COMPLETE ✓")
print("=" * 55)
print(f"  NLP Full — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP Full — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN Full — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN Full — Top-5    : {cnn_metrics['top5_accuracy']}%")
print(f"\n  UPPER BOUND confirmed ✓")
print(f"  Next → Day 6: HAM10000 scarcity experiments")

Building datasets...
✓ Datasets ready
  Train images  : 24527
  Test images   : 6135
  Train batches : 384
  Classes       : 1199
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]



Training CNN — 15 epochs
  Images  : 24527
  Classes : 1199
-------------------------------------------------------
Epoch 01/15 | Loss: 6.9152 | Train Acc: 2.20%
Epoch 02/15 | Loss: 6.1856 | Train Acc: 5.57%
Epoch 03/15 | Loss: 5.7688 | Train Acc: 7.17%
Epoch 04/15 | Loss: 5.4538 | Train Acc: 8.60%
Epoch 05/15 | Loss: 5.1758 | Train Acc: 10.71%
Epoch 06/15 | Loss: 4.8999 | Train Acc: 12.82%
Epoch 07/15 | Loss: 4.6440 | Train Acc: 15.51%
Epoch 08/15 | Loss: 4.3980 | Train Acc: 18.24%
Epoch 09/15 | Loss: 4.1777 | Train Acc: 20.81%
Epoch 10/15 | Loss: 3.9858 | Train Acc: 23.55%
Epoch 11/15 | Loss: 3.8076 | Train Acc: 26.34%
Epoch 12/15 | Loss: 3.6767 | Train Acc: 28.44%
Epoch 13/15 | Loss: 3.5891 | Train Acc: 29.75%
Epoch 14/15 | Loss: 3.5266 | Train Acc: 30.84%
Epoch 15/15 | Loss: 3.4855 | Train Acc: 31.57%
-------------------------------------------------------
✓ Training complete

Evaluating...

EXP 3 — CNN UPPER BOUND (Full Dataset)
  Accuracy     : 9.91%
  F1 Macro     : 2.47%
  F1 

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 64 vs 0

In [10]:
import shutil
import os
import json

# Check space
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free space: {free//(1024**3)} GB")

# Save only metrics (tiny file) — skip full model to save space
nlp_metrics = {
    "accuracy": 16.74, "f1_macro": 2.71,
    "f1_weighted": 10.99, "top5_accuracy": 35.81,
    "total_samples": 7263
}

cnn_metrics = {
    "accuracy"     : 9.91,
    "f1_macro"     : 2.47,
    "f1_weighted"  : 7.2,
    "top5_accuracy": 26.24,
    "total_samples": 6135
}

# Save just the summary JSON — very small
os.makedirs("/kaggle/working/results", exist_ok=True)

summary = {
    "day"        : 5,
    "experiment" : "exp3_full_upperbound",
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "status"     : "Day 5 complete"
}

with open("/kaggle/working/results/day5_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✓ Results saved")

print("\n" + "=" * 55)
print("DAY 5 COMPLETE ✓")
print("=" * 55)
print(f"  NLP Full — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP Full — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN Full — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN Full — Top-5    : {cnn_metrics['top5_accuracy']}%")
print(f"\n  UPPER BOUND confirmed ✓")
print(f"  Next → Day 6: HAM10000 scarcity experiments")

Free space: 0 GB


OSError: [Errno 28] No space left on device

In [11]:
import os
import shutil

# Delete the ZIP file — biggest space user
files_to_delete = [
    "/kaggle/working/zebramap_final.zip",
    "/kaggle/working/data/zebramap_final.zip"
]

for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ Deleted: {f}")

# Check space now
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nFree space now: {free//(1024**3)} GB")

✓ Deleted: /kaggle/working/zebramap_final.zip

Free space now: 9 GB


In [12]:
import json

os.makedirs("/kaggle/working/results", exist_ok=True)

summary = {
    "day": 5,
    "experiment": "exp3_full_upperbound",
    "nlp_metrics": {
        "accuracy": 16.74, "f1_macro": 2.71,
        "f1_weighted": 10.99, "top5_accuracy": 35.81
    },
    "cnn_metrics": {
        "accuracy": 9.91, "f1_macro": 2.47,
        "f1_weighted": 7.2, "top5_accuracy": 26.24
    },
    "status": "Day 5 complete"
}

with open("/kaggle/working/results/day5_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✓ Day 5 summary saved")
print("\nDAY 5 COMPLETE ✓")
print(f"  NLP Full — Accuracy : 16.74%  Top-5: 35.81%")
print(f"  CNN Full — Accuracy : 9.91%   Top-5: 26.24%")
print(f"  Next → Day 6")

✓ Day 5 summary saved

DAY 5 COMPLETE ✓
  NLP Full — Accuracy : 16.74%  Top-5: 35.81%
  CNN Full — Accuracy : 9.91%   Top-5: 26.24%
  Next → Day 6
